This notebook is used to parse sentences from BNC

In [1]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2026-03-09T19:03:08.182459+11:00

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.6.0

Compiler    : Clang 16.0.0 (clang-1600.0.26.6)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit




# Extract sentences that contain animal names

In [2]:
import os
from pathlib import Path
import re
import pandas as pd
import xml.etree.ElementTree as ET
from nltk.stem import WordNetLemmatizer

# Paths
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

target_path = Path(project_folder) / "data" / "dedeyne_feat.csv"
texts_root  = Path(project_folder) / "rawdata" / "BNC" / "download" / "Texts"
out_csv     = Path(project_folder) / "data" / "bnc_matches" / "animals.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)

# Prepare the list of target words
fe = pd.read_csv(target_path)
fe = (fe[["concept", "category", "domain"]].dropna(subset=["concept"]))
fe = fe.loc[(fe["concept"] != fe["category"]) & (fe["category"] != fe["domain"])]
fe = (fe.groupby(["concept", "domain"])["category"]
      .agg(lambda s: " or ".join(sorted(s.dropna().astype(str).unique())))
      .reset_index())
fe = fe.drop_duplicates().reset_index(drop=True)

lemmatizer = WordNetLemmatizer()

def remove_parens(s: str) -> str:
    return s.replace("(", "").replace(")", "")

def extract_head(token: str) -> str:
    token = remove_parens(token)
    last = token.split("_")[-1]       
    head = last.lower()                
    return lemmatizer.lemmatize(head, pos="n")

def display_form(token: str) -> str:
    token = remove_parens(token)
    return token.replace("_", " ")

fe["concept_lem"] = fe["concept"].str.lower().apply(lambda x: lemmatizer.lemmatize(x, pos="n"))
fe["target_word"] = fe["concept_lem"].apply(display_form)        
fe["head_word"]   = fe["concept_lem"].apply(extract_head)       
df_targets = fe[["concept", "category", "domain", "target_word", "head_word"]].drop_duplicates().reset_index(drop=True)

# need to change these categories to align with Gemini reasoning
category_fixes = {
    "dolphin": "mammals",
    "frog": "amphibians",
    "leech": "invertebrates",
    "spider": "invertebrates",
    "toad": "amphibians",
    "whale": "mammals",
    "worm": "invertebrates",
    "orca": "mammals",
    "salamander": "amphibians",
    "sperm whale": "mammals",
    "apron": "kitchen linens",
    "tank": "vehicles",
    "towel": "kitchen linens"
}

for tw, new_cat in category_fixes.items():
    df_targets.loc[df_targets["target_word"] == tw, "category"] = new_cat

# Exclude words of which plural form matches its singular form or its default form is plural
PLURALS = {
    "bison", "deer", "sheep",
    "trout", "salmon", "cod", "carp", "pike", "herring",
    "goldfish", "flatfish", "swordfish",
    "squid", "plaice", "sole", "ray",
    "pant", "pyjama", "jean", "short", "dungaree", 
    "panty", "tongs", "scissors"
}

# Create a list of target words for animal domain
animals_df = df_targets[df_targets["domain"].str.lower() == "animals"].copy()
assert animals_df["target_word"].nunique() == 129 
print(f"The unique number of target words in animal domain is {animals_df['target_word'].nunique()}")

animals_df = animals_df[~animals_df["head_word"].isin(PLURALS)]
print(f"The unique number of target words after excluding zero plurals is {animals_df['target_word'].nunique()}")

animal_heads_set = set(
    animals_df["head_word"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
print(f"The unique number of head words in final dataframe is {len(animal_heads_set)}")

# Create a list of compounds and single words for a given head word
# We will use this to search for target compounds words in BNC sentences
def pattern_tokens(s: str):
    return [tok for tok in s.lower().split() if tok]

head_to_patterns = {}
for _, row in animals_df.iterrows():
    head = row["head_word"].lower().strip()
    tw   = row["target_word"].strip()
    toks = pattern_tokens(tw)
    if not toks:
        continue
    head_to_patterns.setdefault(head, set()).add(tuple(toks))
for h in list(head_to_patterns.keys()):
    head_to_patterns[h] = sorted(list(head_to_patterns[h]), key=lambda t: (-len(t), t))

# We will include only common nouns
def is_common_noun(c5: str) -> bool:
    if not c5:
        return False
    if "NP0" in c5:  # exclude proper noun
        return False
    return any(p.startswith("NN") for p in c5.split("-"))  # if any of the tags include common nouns

def parse_bnc_file(xml_path: Path):
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    sentences = []
    for s_el in root.findall(".//s"):
        sent_id = s_el.get("n")
        tokens = []
        for child in list(s_el):
            tag = child.tag.lower()
            if tag == "w":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                hw   = (child.get("hw") or "").strip().lower()
                c5   = (child.get("c5") or "").strip()
                tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
            elif tag == "c":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
            else:
                for g in child.iter():
                    gtag = g.tag.lower()
                    if gtag == "w":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        hw   = (g.get("hw") or "").strip().lower()
                        c5   = (g.get("c5") or "").strip()
                        tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
                    elif gtag == "c":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
        surface = "".join(t["text_withoutstrip"] for t in tokens).strip()
        sentences.append({"sent_id": sent_id, "tokens": tokens, "surface": surface})
    return sentences

def highlight_span(sent, span):
    i0, i1 = span
    tokens = sent["tokens"]

    new_surfs = []
    for i, t in enumerate(tokens):
        s = t["text_withoutstrip"]
        if t["kind"] == "w" and i0 <= i <= i1:
            if i == i0 and i == i1:
                # single-word target
                s = f"**{t['text']}**" + (" " if t["text_withoutstrip"].endswith(" ") else "")
            elif i == i0:
                # first word of multi-word target
                s = f"**{t['text']}"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            elif i == i1:
                # last word of multi-word target
                s = f"{t['text']}**"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            else:
                # middle word(s) of multi-word target
                s = t["text_withoutstrip"]
        new_surfs.append(s)

    sentence = "".join(new_surfs).strip()
    return sentence

def find_matches_with_compounds(sent):
    tokens = sent["tokens"]
    matches = []

    word_positions = [i for i, t in enumerate(tokens) if t["kind"] == "w"]
    abs_to_word_i = {abs_i: wi for wi, abs_i in enumerate(word_positions)}

    for abs_i, tok in enumerate(tokens):
        if tok.get("kind") != "w":
            continue

        hw = tok.get("hw", "")
        c5 = tok.get("c5", "")
        surf = tok.get("text", "")

        if hw not in animal_heads_set:
            continue
        if not is_common_noun(c5):
            continue
        if surf != hw:
            continue

        patterns = head_to_patterns.get(hw, [])
        has_single = any(len(p) == 1 for p in patterns)
        has_multi  = any(len(p) > 1 for p in patterns)

        compound_found = None

        # Try to match the longest multi-word pattern
        if has_multi and abs_i in abs_to_word_i:
            wi = abs_to_word_i[abs_i]
            for patt in patterns:                   # longest first (already sorted)
                if len(patt) == 1:
                    continue                       # skip singles here
                L = len(patt)
                if wi - (L - 1) < 0:
                    continue

                word_abs_span = word_positions[wi - (L - 1): wi + 1]

                # must be contiguous
                if any((b - a) != 1 for a, b in zip(word_abs_span, word_abs_span[1:])):
                    continue

                if [tokens[j]["text"] for j in word_abs_span] == list(patt):
                    compound_found = {
                        "span": (word_abs_span[0], word_abs_span[-1]),
                        "target_word": " ".join(patt),
                        "head": hw
                    }
                    break

        if compound_found:
            matches.append(compound_found)

        else:
            # Only allow bare match if head has a 1-token target
            if has_single:
                matches.append({
                    "span": (abs_i, abs_i),
                    "target_word": surf,
                    "head": hw
                })

    return matches

# Run over BNC files and save
rows = []

for xml_path in texts_root.rglob("*.xml"):
    rel = xml_path.stem

    try:
        # Parse the XML once, so we can extract doc_type here
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Extract document type
        wtext = root.find(".//wtext")
        if wtext is None:
            wtext = root.find(".//stext")
        doc_type = wtext.get("type") if wtext is not None else None

        # Call your sentence parsing function
        sents = parse_bnc_file(xml_path)

    except ET.ParseError:
        continue

    for idx, sent in enumerate(sents):
        matches = find_matches_with_compounds(sent)
        if not matches:
            continue
        prev_sentence = sents[idx-1]["surface"] if idx-1 >= 0 else ""
        next_sentence = sents[idx+1]["surface"] if idx+1 < len(sents) else ""
        for m in matches:
            sentence_highlighted = highlight_span(sent, m["span"])
            rows.append({
                "doc_type": doc_type,
                "file_path": rel,
                "sentence_id": sent["sent_id"] if sent["sent_id"] is not None else str(idx+1),
                "previous_sentence": prev_sentence,
                "sentence": sentence_highlighted,
                "next_sentence": next_sentence,
                "target_word": m["target_word"],
            })

out_df = pd.DataFrame(rows, columns=[
    "doc_type", "file_path", "sentence_id", "previous_sentence", "sentence", "next_sentence", "target_word"
])

df_targets_small = (df_targets[["concept", "category", "target_word"]].drop_duplicates())
out_df = out_df.merge(df_targets_small, on="target_word", how="left")

out_df.to_csv(out_csv, index=False)
print(f"Saved {len(out_df):,} rows to output file.")

The unique number of target words in animal domain is 129
The unique number of target words after excluding zero plurals is 113
The unique number of head words in final dataframe is 109
Saved 46,339 rows to output file.


# Extract sentences that contain artifact names

In [3]:
import os
from pathlib import Path
import re
import pandas as pd
import xml.etree.ElementTree as ET
from nltk.stem import WordNetLemmatizer

# Paths
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

target_path = Path(project_folder) / "data" / "dedeyne_feat.csv"
texts_root  = Path(project_folder) / "rawdata" / "BNC" / "download" / "Texts"
out_csv     = Path(project_folder) / "data" / "bnc_matches" / "artifacts.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)

# Prepare the list of target words
fe = pd.read_csv(target_path)
fe = (fe[["concept", "category", "domain"]].dropna(subset=["concept"]))
fe = fe.loc[(fe["concept"] != fe["category"]) & (fe["category"] != fe["domain"])]
fe = (fe.groupby(["concept", "domain"])["category"]
      .agg(lambda s: " or ".join(sorted(s.dropna().astype(str).unique())))
      .reset_index())
fe = fe.drop_duplicates().reset_index(drop=True)

lemmatizer = WordNetLemmatizer()

def remove_parens(s: str) -> str:
    return s.replace("(", "").replace(")", "")

def extract_head(token: str) -> str:
    token = remove_parens(token)
    last = token.split("_")[-1]       
    head = last.lower()                
    return lemmatizer.lemmatize(head, pos="n")

def display_form(token: str) -> str:
    token = remove_parens(token)
    return token.replace("_", " ")

fe["concept_lem"] = fe["concept"].str.lower().apply(lambda x: lemmatizer.lemmatize(x, pos="n"))
fe["target_word"] = fe["concept_lem"].apply(display_form)        
fe["head_word"]   = fe["concept_lem"].apply(extract_head)       
df_targets = fe[["concept", "category", "domain", "target_word", "head_word"]].drop_duplicates().reset_index(drop=True)

# need to change these categories to align with Gemini reasoning
category_fixes = {
    "dolphin": "mammals",
    "frog": "amphibians",
    "leech": "invertebrates",
    "spider": "invertebrates",
    "toad": "amphibians",
    "whale": "mammals",
    "worm": "invertebrates",
    "orca": "mammals",
    "salamander": "amphibians",
    "sperm whale": "mammals",
    "apron": "kitchen linens",
    "tank": "vehicles",
    "towel": "kitchen linens"
}

for tw, new_cat in category_fixes.items():
    df_targets.loc[df_targets["target_word"] == tw, "category"] = new_cat

# Exclude words of which plural form matches its singular form or its default form is plural
PLURALS = {
    "bison", "deer", "sheep",
    "trout", "salmon", "cod", "carp", "pike", "herring",
    "goldfish", "flatfish", "swordfish",
    "squid", "plaice", "sole", "ray",
    "pant", "pyjama", "jean", "short", "dungaree", 
    "panty", "tongs", "scissors"
}

# Create a list of target words for artifact domain
artifacts_df = df_targets[df_targets["domain"].str.lower() == "artifacts"].copy()
assert artifacts_df["target_word"].nunique() == 162
print(f"The unique number of target words in artifact domain is {artifacts_df['target_word'].nunique()}")

artifacts_df = artifacts_df[~artifacts_df["head_word"].isin(PLURALS)]
print(f"The unique number of target words after excluding plurals by default is {artifacts_df['target_word'].nunique()}")

artifact_heads_set = set(
    artifacts_df["head_word"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
print(f"The unique number of head words in final dataframe is {len(artifact_heads_set)}")

# Create a list of compounds and single words for a given head word
# We will use this to search for target compounds words in BNC sentences
def pattern_tokens(s: str):
    return [tok for tok in s.lower().split() if tok]

head_to_patterns = {}
for _, row in artifacts_df.iterrows():
    head = row["head_word"].lower().strip()
    tw   = row["target_word"].strip()
    toks = pattern_tokens(tw)
    if not toks:
        continue
    head_to_patterns.setdefault(head, set()).add(tuple(toks))
for h in list(head_to_patterns.keys()):
    head_to_patterns[h] = sorted(list(head_to_patterns[h]), key=lambda t: (-len(t), t))

# We will include only common nouns
def is_common_noun(c5: str) -> bool:
    if not c5:
        return False
    if "NP0" in c5:  # exclude proper noun
        return False
    return any(p.startswith("NN") for p in c5.split("-"))  # if any of the tags include common nouns

def parse_bnc_file(xml_path: Path):
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    sentences = []
    for s_el in root.findall(".//s"):
        sent_id = s_el.get("n")
        tokens = []
        for child in list(s_el):
            tag = child.tag.lower()
            if tag == "w":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                hw   = (child.get("hw") or "").strip().lower()
                c5   = (child.get("c5") or "").strip()
                tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
            elif tag == "c":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
            else:
                for g in child.iter():
                    gtag = g.tag.lower()
                    if gtag == "w":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        hw   = (g.get("hw") or "").strip().lower()
                        c5   = (g.get("c5") or "").strip()
                        tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
                    elif gtag == "c":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
        surface = "".join(t["text_withoutstrip"] for t in tokens).strip()
        sentences.append({"sent_id": sent_id, "tokens": tokens, "surface": surface})
    return sentences

def highlight_span(sent, span):
    i0, i1 = span
    tokens = sent["tokens"]

    new_surfs = []
    for i, t in enumerate(tokens):
        s = t["text_withoutstrip"]
        if t["kind"] == "w" and i0 <= i <= i1:
            if i == i0 and i == i1:
                # single-word target
                s = f"**{t['text']}**" + (" " if t["text_withoutstrip"].endswith(" ") else "")
            elif i == i0:
                # first word of multi-word target
                s = f"**{t['text']}"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            elif i == i1:
                # last word of multi-word target
                s = f"{t['text']}**"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            else:
                # middle word(s) of multi-word target
                s = t["text_withoutstrip"]
        new_surfs.append(s)

    sentence = "".join(new_surfs).strip()
    return sentence

def find_matches_with_compounds(sent):
    tokens = sent["tokens"]
    matches = []

    word_positions = [i for i, t in enumerate(tokens) if t["kind"] == "w"]
    abs_to_word_i = {abs_i: wi for wi, abs_i in enumerate(word_positions)}

    for abs_i, tok in enumerate(tokens):
        if tok.get("kind") != "w":
            continue

        hw = tok.get("hw", "")
        c5 = tok.get("c5", "")
        surf = tok.get("text", "")

        if hw not in artifact_heads_set:
            continue
        if not is_common_noun(c5):
            continue
        if surf != hw:
            continue

        patterns = head_to_patterns.get(hw, [])
        has_single = any(len(p) == 1 for p in patterns)
        has_multi  = any(len(p) > 1 for p in patterns)

        compound_found = None

        # Try to match the longest multi-word pattern
        if has_multi and abs_i in abs_to_word_i:
            wi = abs_to_word_i[abs_i]
            for patt in patterns:                   # longest first (already sorted)
                if len(patt) == 1:
                    continue                       # skip singles here
                L = len(patt)
                if wi - (L - 1) < 0:
                    continue

                word_abs_span = word_positions[wi - (L - 1): wi + 1]

                # must be contiguous
                if any((b - a) != 1 for a, b in zip(word_abs_span, word_abs_span[1:])):
                    continue

                if [tokens[j]["text"] for j in word_abs_span] == list(patt):
                    compound_found = {
                        "span": (word_abs_span[0], word_abs_span[-1]),
                        "target_word": " ".join(patt),
                        "head": hw
                    }
                    break

        if compound_found:
            matches.append(compound_found)

        else:
            # Only allow bare match if head has a 1-token target
            if has_single:
                matches.append({
                    "span": (abs_i, abs_i),
                    "target_word": surf,
                    "head": hw
                })

    return matches

# Run over BNC files and save
rows = []

for xml_path in texts_root.rglob("*.xml"):
    rel = xml_path.stem

    try:
        # Parse the XML once, so we can extract doc_type here
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Extract document type
        wtext = root.find(".//wtext")
        if wtext is None:
            wtext = root.find(".//stext")
        doc_type = wtext.get("type") if wtext is not None else None

        # Call your sentence parsing function
        sents = parse_bnc_file(xml_path)

    except ET.ParseError:
        continue

    for idx, sent in enumerate(sents):
        matches = find_matches_with_compounds(sent)
        if not matches:
            continue
        prev_sentence = sents[idx-1]["surface"] if idx-1 >= 0 else ""
        next_sentence = sents[idx+1]["surface"] if idx+1 < len(sents) else ""
        for m in matches:
            sentence_highlighted = highlight_span(sent, m["span"])
            rows.append({
                "doc_type": doc_type,
                "file_path": rel,
                "sentence_id": sent["sent_id"] if sent["sent_id"] is not None else str(idx+1),
                "previous_sentence": prev_sentence,
                "sentence": sentence_highlighted,
                "next_sentence": next_sentence,
                "target_word": m["target_word"],
            })

out_df = pd.DataFrame(rows, columns=[
    "doc_type", "file_path", "sentence_id", "previous_sentence", "sentence", "next_sentence", "target_word"
])

df_targets_small = (df_targets[["concept", "category", "target_word"]].drop_duplicates())
out_df = out_df.merge(df_targets_small, on="target_word", how="left")

out_df.to_csv(out_csv, index=False)
print(f"Saved {len(out_df):,} rows to output file.")

The unique number of target words in artifact domain is 162
The unique number of target words after excluding plurals by default is 154
The unique number of head words in final dataframe is 145
Saved 206,462 rows to output file.


# Sample sentences from the extracted sentences

In [4]:
import os 
import pandas as pd
from pathlib import Path
import numpy as np

project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

animal_path = Path(project_folder) / "data" / "bnc_matches" / "animals.csv"
artifact_path = Path(project_folder) / "data" / "bnc_matches" / "artifacts.csv"

animals = pd.read_csv(animal_path)
print("The total number of sentences in animal domain:", len(animals))
artifacts = pd.read_csv(artifact_path, low_memory=False)
print("The total number of sentences in artifact domain:", len(artifacts))

The total number of sentences in animal domain: 46339
The total number of sentences in artifact domain: 206462


In [5]:
animal_spoken_target = 5000
artifact_spoken_target = 10000
written_target = 10000

spoken_types = ["OTHERSP", "CONVRSN"]

def make_sample(df, total_target):
    counts = (
        df.groupby("target_word")
          .size()
          .reset_index(name="n")
    )

    counts["total"]   = counts["n"].sum()
    counts["p"]       = counts["n"] / counts["total"]
    counts["target"]  = counts["p"] * total_target
    counts["ceiling"] = np.ceil(counts["target"]).astype(int)
    counts["floor"]   = np.where(counts["ceiling"] < 10, 10, counts["ceiling"])
    counts["sample"]  = np.where(counts["n"] < counts["floor"], counts["n"], counts["floor"])

    return counts

animals_spoken = animals[animals["doc_type"].isin(spoken_types)]
animals_written = animals[~animals["doc_type"].isin(spoken_types)]

animal_spoken_sample = make_sample(animals_spoken,animal_spoken_target)
assert abs(animal_spoken_sample["target"].sum() - 5000) < 1e-6
animal_written_sample = make_sample(animals_written,written_target)
assert abs(animal_written_sample["target"].sum() - 10000) < 1e-6

artifacts_spoken = artifacts[artifacts["doc_type"].isin(spoken_types)]
artifacts_written = artifacts[~artifacts["doc_type"].isin(spoken_types)]

artifact_spoken_sample = make_sample(artifacts_spoken,artifact_spoken_target)
assert abs(artifact_spoken_sample["target"].sum() - 10000) < 1e-6
artifact_written_sample = make_sample(artifacts_written,written_target)
assert abs(artifact_written_sample["target"].sum() - 10000) < 1e-6

In [6]:
def stratified_sample_by_target(df, sample_df, seed=123):
    df_with_sizes = df.merge(
        sample_df[["target_word", "n", "sample"]],
        on="target_word",
        how="left"
    )
    sampled = (
        df_with_sizes
        .groupby("target_word", group_keys=False)
        .apply(lambda g: g.sample(
            n=int(g["sample"].iloc[0]),
            random_state=seed,
            replace=False
        ))
        .reset_index(drop=True)
    )
    sampled = sampled.drop(columns=["sample"])
    return sampled

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    sampled_spoken_animal = stratified_sample_by_target(animals_spoken, animal_spoken_sample, seed=123)
    sampled_written_animal = stratified_sample_by_target(animals_written, animal_written_sample, seed=123)
    sampled_animal = pd.concat([sampled_spoken_animal, sampled_written_animal],ignore_index=True)

out_path = Path(project_folder) / "data" / "bnc_matches" / "animals_sampled.csv"
sampled_animal.to_csv(out_path, index=False)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    sampled_spoken_artifact = stratified_sample_by_target(artifacts_spoken, artifact_spoken_sample, seed=123)
    sampled_written_artifact = stratified_sample_by_target(artifacts_written, artifact_written_sample, seed=123)
    sampled_artifact = pd.concat([sampled_spoken_artifact, sampled_written_artifact],ignore_index=True)

out_path = Path(project_folder) / "data" / "bnc_matches" / "artifacts_sampled.csv"
sampled_artifact.to_csv(out_path, index=False)

print("The number of samples in animal domain:", len(sampled_animal))
print("The number of samples in artifact domain:", len(sampled_artifact))

The number of samples in animal domain: 15169
The number of samples in artifact domain: 20590


# Extract sentences that contain "flea"

We realised that there was a typo "flee" in Leuven data after running our word sense disambiguation step, so we will extract sentences that contain "flea" and sample from them.

In [7]:
import os
from pathlib import Path
import re
import pandas as pd
import xml.etree.ElementTree as ET
from nltk.stem import WordNetLemmatizer

# Paths
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

target_path = Path(project_folder) / "data" / "dedeyne_feat.csv"
texts_root  = Path(project_folder) / "rawdata" / "BNC" / "download" / "Texts"
out_csv     = Path(project_folder) / "data" / "bnc_matches" / "animals_flea.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)

# Prepare the list of target words
fe = pd.read_csv(target_path)

# Fix typo for flea
fe.loc[fe["concept"] == "flee", "concept"] = "flea"

fe = (fe[["concept", "category", "domain"]].dropna(subset=["concept"]))
fe = fe.loc[(fe["concept"] != fe["category"]) & (fe["category"] != fe["domain"])]
fe = (fe.groupby(["concept", "domain"])["category"]
      .agg(lambda s: " or ".join(sorted(s.dropna().astype(str).unique())))
      .reset_index())
fe = fe.drop_duplicates().reset_index(drop=True)

lemmatizer = WordNetLemmatizer()

def remove_parens(s: str) -> str:
    return s.replace("(", "").replace(")", "")

def extract_head(token: str) -> str:
    token = remove_parens(token)
    last = token.split("_")[-1]       
    head = last.lower()                
    return lemmatizer.lemmatize(head, pos="n")

def display_form(token: str) -> str:
    token = remove_parens(token)
    return token.replace("_", " ")

fe["concept_lem"] = fe["concept"].str.lower().apply(lambda x: lemmatizer.lemmatize(x, pos="n"))
fe["target_word"] = fe["concept_lem"].apply(display_form)        
fe["head_word"]   = fe["concept_lem"].apply(extract_head)       
df_targets = fe[["concept", "category", "domain", "target_word", "head_word"]].drop_duplicates().reset_index(drop=True)

# Keep only flea
df_targets = df_targets[df_targets["concept"] == "flea"].reset_index(drop=True)

# need to change these categories to align with Gemini reasoning
category_fixes = {
    "dolphin": "mammals",
    "frog": "amphibians",
    "leech": "invertebrates",
    "spider": "invertebrates",
    "toad": "amphibians",
    "whale": "mammals",
    "worm": "invertebrates",
    "orca": "mammals",
    "salamander": "amphibians",
    "sperm whale": "mammals",
    "apron": "kitchen linens",
    "tank": "vehicles",
    "towel": "kitchen linens"
}

for tw, new_cat in category_fixes.items():
    df_targets.loc[df_targets["target_word"] == tw, "category"] = new_cat

# Exclude words of which plural form matches its singular form or its default form is plural
PLURALS = {
    "bison", "deer", "sheep",
    "trout", "salmon", "cod", "carp", "pike", "herring",
    "goldfish", "flatfish", "swordfish",
    "squid", "plaice", "sole", "ray",
    "pant", "pyjama", "jean", "short", "dungaree", 
    "panty", "tongs", "scissors"
}

# Create a list of target words for animal domain
animals_df = df_targets[df_targets["domain"].str.lower() == "animals"].copy()
assert animals_df["target_word"].nunique() == 1
print(f"The unique number of target words in animal domain is {animals_df['target_word'].nunique()}")

animals_df = animals_df[~animals_df["head_word"].isin(PLURALS)]
print(f"The unique number of target words after excluding zero plurals is {animals_df['target_word'].nunique()}")

animal_heads_set = set(
    animals_df["head_word"].dropna().astype(str).str.strip().str.lower().drop_duplicates().tolist()
)
print(f"The unique number of head words in final dataframe is {len(animal_heads_set)}")

# Create a list of compounds and single words for a given head word
# We will use this to search for target compounds words in BNC sentences
def pattern_tokens(s: str):
    return [tok for tok in s.lower().split() if tok]

head_to_patterns = {}
for _, row in animals_df.iterrows():
    head = row["head_word"].lower().strip()
    tw   = row["target_word"].strip()
    toks = pattern_tokens(tw)
    if not toks:
        continue
    head_to_patterns.setdefault(head, set()).add(tuple(toks))
for h in list(head_to_patterns.keys()):
    head_to_patterns[h] = sorted(list(head_to_patterns[h]), key=lambda t: (-len(t), t))

# We will include only common nouns
def is_common_noun(c5: str) -> bool:
    if not c5:
        return False
    if "NP0" in c5:  # exclude proper noun
        return False
    return any(p.startswith("NN") for p in c5.split("-"))  # if any of the tags include common nouns

def parse_bnc_file(xml_path: Path):
    tree = ET.parse(str(xml_path))
    root = tree.getroot()
    sentences = []
    for s_el in root.findall(".//s"):
        sent_id = s_el.get("n")
        tokens = []
        for child in list(s_el):
            tag = child.tag.lower()
            if tag == "w":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                hw   = (child.get("hw") or "").strip().lower()
                c5   = (child.get("c5") or "").strip()
                tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
            elif tag == "c":
                text_withoutstrip = (child.text or "")
                text = (child.text or "").strip()
                tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
            else:
                for g in child.iter():
                    gtag = g.tag.lower()
                    if gtag == "w":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        hw   = (g.get("hw") or "").strip().lower()
                        c5   = (g.get("c5") or "").strip()
                        tokens.append({"kind":"w","text_withoutstrip":text_withoutstrip,"text":text,"hw":hw,"c5":c5})
                    elif gtag == "c":
                        text_withoutstrip = (g.text or "")
                        text = (g.text or "").strip()
                        tokens.append({"kind":"c","text_withoutstrip":text_withoutstrip,"text":text})
        surface = "".join(t["text_withoutstrip"] for t in tokens).strip()
        sentences.append({"sent_id": sent_id, "tokens": tokens, "surface": surface})
    return sentences

def highlight_span(sent, span):
    i0, i1 = span
    tokens = sent["tokens"]

    new_surfs = []
    for i, t in enumerate(tokens):
        s = t["text_withoutstrip"]
        if t["kind"] == "w" and i0 <= i <= i1:
            if i == i0 and i == i1:
                # single-word target
                s = f"**{t['text']}**" + (" " if t["text_withoutstrip"].endswith(" ") else "")
            elif i == i0:
                # first word of multi-word target
                s = f"**{t['text']}"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            elif i == i1:
                # last word of multi-word target
                s = f"{t['text']}**"
                if t["text_withoutstrip"].endswith(" "):
                    s += " "
            else:
                # middle word(s) of multi-word target
                s = t["text_withoutstrip"]
        new_surfs.append(s)

    sentence = "".join(new_surfs).strip()
    return sentence

def find_matches_with_compounds(sent):
    tokens = sent["tokens"]
    matches = []

    word_positions = [i for i, t in enumerate(tokens) if t["kind"] == "w"]
    abs_to_word_i = {abs_i: wi for wi, abs_i in enumerate(word_positions)}

    for abs_i, tok in enumerate(tokens):
        if tok.get("kind") != "w":
            continue

        hw = tok.get("hw", "")
        c5 = tok.get("c5", "")
        surf = tok.get("text", "")

        if hw not in animal_heads_set:
            continue
        if not is_common_noun(c5):
            continue
        if surf != hw:
            continue

        patterns = head_to_patterns.get(hw, [])
        has_single = any(len(p) == 1 for p in patterns)
        has_multi  = any(len(p) > 1 for p in patterns)

        compound_found = None

        # Try to match the longest multi-word pattern
        if has_multi and abs_i in abs_to_word_i:
            wi = abs_to_word_i[abs_i]
            for patt in patterns:                   # longest first (already sorted)
                if len(patt) == 1:
                    continue                       # skip singles here
                L = len(patt)
                if wi - (L - 1) < 0:
                    continue

                word_abs_span = word_positions[wi - (L - 1): wi + 1]

                # must be contiguous
                if any((b - a) != 1 for a, b in zip(word_abs_span, word_abs_span[1:])):
                    continue

                if [tokens[j]["text"] for j in word_abs_span] == list(patt):
                    compound_found = {
                        "span": (word_abs_span[0], word_abs_span[-1]),
                        "target_word": " ".join(patt),
                        "head": hw
                    }
                    break

        if compound_found:
            matches.append(compound_found)

        else:
            # Only allow bare match if head has a 1-token target
            if has_single:
                matches.append({
                    "span": (abs_i, abs_i),
                    "target_word": surf,
                    "head": hw
                })

    return matches

# Run over BNC files and save
rows = []

for xml_path in texts_root.rglob("*.xml"):
    rel = xml_path.stem

    try:
        # Parse the XML once, so we can extract doc_type here
        tree = ET.parse(xml_path)
        root = tree.getroot()

        # Extract document type
        wtext = root.find(".//wtext")
        if wtext is None:
            wtext = root.find(".//stext")
        doc_type = wtext.get("type") if wtext is not None else None

        # Call your sentence parsing function
        sents = parse_bnc_file(xml_path)

    except ET.ParseError:
        continue

    for idx, sent in enumerate(sents):
        matches = find_matches_with_compounds(sent)
        if not matches:
            continue
        prev_sentence = sents[idx-1]["surface"] if idx-1 >= 0 else ""
        next_sentence = sents[idx+1]["surface"] if idx+1 < len(sents) else ""
        for m in matches:
            sentence_highlighted = highlight_span(sent, m["span"])
            rows.append({
                "doc_type": doc_type,
                "file_path": rel,
                "sentence_id": sent["sent_id"] if sent["sent_id"] is not None else str(idx+1),
                "previous_sentence": prev_sentence,
                "sentence": sentence_highlighted,
                "next_sentence": next_sentence,
                "target_word": m["target_word"],
            })

out_df = pd.DataFrame(rows, columns=[
    "doc_type", "file_path", "sentence_id", "previous_sentence", "sentence", "next_sentence", "target_word"
])

df_targets_small = (df_targets[["concept", "category", "target_word"]].drop_duplicates())
out_df = out_df.merge(df_targets_small, on="target_word", how="left")

out_df.to_csv(out_csv, index=False)
print(f"Saved {len(out_df):,} rows to output file.")

The unique number of target words in animal domain is 1
The unique number of target words after excluding zero plurals is 1
The unique number of head words in final dataframe is 1
Saved 141 rows to output file.


# Sample sentences for "flea"

In [8]:
import os 
import pandas as pd
from pathlib import Path
import numpy as np

spoken_types = ["OTHERSP", "CONVRSN"]

project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)

animal_path = Path(project_folder) / "data" / "bnc_matches" / "animals.csv"
animal_flea_path = Path(project_folder) / "data" / "bnc_matches" / "animals_flea.csv"
animal_sample_path = Path(project_folder) / "data" / "bnc_matches" / "animals_sampled.csv"

animals = pd.read_csv(animal_path)
animals_flea = pd.read_csv(animal_flea_path)

animals_flea_spoken = animals_flea[animals_flea["doc_type"].isin(spoken_types)]
animals_flea_written = animals_flea[~animals_flea["doc_type"].isin(spoken_types)]

print("The number of sentences with flea in spoken corpus:", len(animals_flea_spoken))
print("The number of sentences with flea in written corpus:", len(animals_flea_written))

animals_all = pd.concat([animals, animals_flea], ignore_index=True)
animal_sample = pd.read_csv(animal_sample_path)

def make_sample_flea(df, animal_sample, spoken=True, type_col="doc_type"):

    if spoken:
        df_filtered = df[df[type_col].isin(spoken_types)]
        sample_filtered = animal_sample[animal_sample[type_col].isin(spoken_types)]
    else:
        df_filtered = df[~df[type_col].isin(spoken_types)]
        sample_filtered = animal_sample[~animal_sample[type_col].isin(spoken_types)]

    counts = (
        df_filtered.groupby("target_word")
                   .size()
                   .reset_index(name="n")
    )
    counts["total"] = counts["n"].sum()
    counts["p"] = counts["n"] / counts["total"]

    flea_p = counts.loc[counts["target_word"] == "flea", "p"]
    flea_p = float(flea_p)

    total_s = len(sample_filtered)

    flea_sample = int(round(flea_p * total_s / (1 - flea_p)))

    return flea_sample

with warnings.catch_warnings():
    warnings.simplefilter("ignore", FutureWarning)
    flea_spoken  = make_sample_flea(animals_all, animal_sample, spoken=True)
    flea_written = make_sample_flea(animals_all, animal_sample, spoken=False)

print("The number of samples with flea in spoken corpus:", flea_spoken)
print("The number of samples with flea in written corpus:", flea_written)

The number of sentences with flea in spoken corpus: 8
The number of sentences with flea in written corpus: 133
The number of samples with flea in spoken corpus: 8
The number of samples with flea in written corpus: 33


In [9]:
def sample_rows(df, sample_number, seed=123):
    return df.sample(
        n=sample_number, 
        random_state=seed, 
        replace=False)

sampled_spoken_flea = sample_rows(animals_flea_spoken, flea_spoken, seed=123)
sampled_written_flea = sample_rows(animals_flea_written, flea_written, seed=123)

sampled_flea = pd.concat([sampled_spoken_flea, sampled_written_flea], ignore_index=True)
sampled_flea["n"] = len(animals_flea)

out_path = Path(project_folder) / "data" / "bnc_matches" / "animals_sampled_flea.csv"
sampled_flea.to_csv(out_path, index=False)

print("Total number of samples with flea:", len(sampled_flea))

Total number of samples with flea: 41
